# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Checklist: `checklist.md`.

Data notes (from `columns.csv` and inspection): all features (channel, device age, price, daily hours) are known at sale/registration, so they are available at prediction time. `device_serial` is an identifier and is dropped. There is no date column and every serial is unique (checked below), so there is no time order or repeated entity to respect; a stratified random split is appropriate.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_recall_curve
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', int(df.isna().sum().sum()))
print(df.claim_within_year.mean())
df.describe()

## 1. Hold out the test set first
Serials are unique and there is no time column, so a stratified random 20% split. The test set is not touched until step 6.

In [ ]:
CAT=['channel']; NUM=['device_age_years','price_eur','daily_hours']; FEATS=CAT+NUM
train,test=train_test_split(df,test_size=0.2,stratify=df.claim_within_year,random_state=SEED)
Xtr,ytr=train[FEATS],train.claim_within_year
Xte,yte=test[FEATS],test.claim_within_year
print(len(train),len(test),ytr.mean(),yte.mean())

## 2-4. Candidates (named before any scoring)
1. **Baseline**: predicts the training prevalence for everyone.
2. **Logistic regression** (scaled numerics, one-hot channel), grid over `C` in {0.01, 0.1, 1, 10}.
3. **Gradient boosting** (`HistGradientBoostingClassifier`), grid over `max_depth` in {2, 3} and `learning_rate` in {0.03, 0.1}, with `max_iter`=200 and `l2_regularization`=1.

Selection metric: average precision (PR-AUC), since only ~13% of rows are positive. Features are the four columns above for all models. Cross-validation: 5-fold stratified, inside the training rows only (serials are unique, so folds hold whole entities). All preprocessing sits inside the pipelines, so it is fit within each fold. The decision threshold is also chosen on out-of-fold predictions only.

In [ ]:
pre=lambda scale: ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler() if scale else 'passthrough',NUM)])
cands={
 'baseline':(Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),{}),
 'logreg':(Pipeline([('p',pre(True)),('m',LogisticRegression(max_iter=1000))]),{'m__C':[0.01,0.1,1,10]}),
 'hgb':(Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_iter=200,l2_regularization=1.0,random_state=SEED))]),
        {'m__max_depth':[2,3],'m__learning_rate':[0.03,0.1]}),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
searches={}
for n,(pipe,grid) in cands.items():
    g=GridSearchCV(pipe,grid or {'m__strategy':['prior']},scoring='average_precision',cv=cv).fit(Xtr,ytr)
    searches[n]=g; print(f'{n:9s} CV AP={g.best_score_:.4f}  params={g.best_params_}')
print('prevalence (what AP of a no-skill model equals):',round(ytr.mean(),4))

## 5. Choose model and threshold on CV only

In [ ]:
best=max(searches,key=lambda n:searches[n].best_score_)
print('chosen:',best,searches[best].best_params_)
oof=cross_val_predict(searches[best].best_estimator_,Xtr,ytr,cv=StratifiedKFold(5,shuffle=True,random_state=1),method='predict_proba')[:,1]
print('OOF AP',average_precision_score(ytr,oof),'OOF AUC',roc_auc_score(ytr,oof))
p,r,t=precision_recall_curve(ytr,oof); f=2*p*r/(p+r+1e-12)
THR=float(t[np.argmax(f[:-1])]); print('threshold (max OOF F1):',round(THR,4),'OOF F1',f[:-1].max())
print('flag-everyone F1:',f1_score(ytr,np.ones_like(ytr)))

**Decision recorded before test scoring:** the model, its hyperparameters and the threshold are exactly those printed above (best CV average precision; threshold maximising out-of-fold F1). Nothing is changed after seeing test results.

## 6. Refit on all training rows; score the test set once

In [ ]:
final=searches[best].best_estimator_.fit(Xtr,ytr)
pt=final.predict_proba(Xte)[:,1]
base=searches['baseline'].best_estimator_.predict_proba(Xte)[:,1]
res=pd.DataFrame({'AP':[average_precision_score(yte,pt),average_precision_score(yte,base)],
                  'ROC AUC':[roc_auc_score(yte,pt),roc_auc_score(yte,base)]},index=[best,'baseline'])
res.loc[best,'F1@thr']=f1_score(yte,pt>=THR); res.loc['baseline','F1@thr']=f1_score(yte,np.ones_like(yte))
print(res.round(4)); print('test prevalence',round(yte.mean(),4))
rng=np.random.default_rng(SEED); aps=[]
for _ in range(1000):
    i=rng.integers(0,len(yte),len(yte))
    if yte.iloc[i].sum(): aps.append(average_precision_score(yte.iloc[i],pt[i]))
print('bootstrap 95% CI for test AP:',np.percentile(aps,[2.5,97.5]).round(4))

## Conclusion
See the printed test results above: the headline is the test AP / ROC AUC of the chosen model compared with the prevalence baseline. If the AP CI sits clearly above the test prevalence (~0.13) and AUC is well above 0.5, the features carry signal; otherwise claims cannot be predicted from these four sale-time fields.